In [1]:
import os
from anthropic import Anthropic
from dotenv import load_dotenv

In [2]:
load_dotenv()
client=Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
model="claude-sonnet-4-5"

In [3]:
def add_user_message(messages,text):
    user_message={"role":"user", "content":text}
    messages.append(user_message)

def add_assistant_message(messages,text):
    assistant_message={"role":"assistant", "content":text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model":model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }

    if system:
        params["system"] =system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    response = client.messages.create(**params)
    return response.content[0].text

In [4]:
import json
def generate_dataset():
    prompt ="""
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text=chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [5]:
dataset = generate_dataset()
print(dataset)

[{'task': "Write a Python function that parses an AWS ARN (Amazon Resource Name) and returns a dictionary containing the partition, service, region, account-id, and resource components. For example, 'arn:aws:s3:us-east-1:123456789012:bucket/my-bucket' should return {'partition': 'aws', 'service': 's3', 'region': 'us-east-1', 'account_id': '123456789012', 'resource': 'bucket/my-bucket'}."}, {'task': "Create a JSON object that defines an AWS IAM policy allowing read-only access to a specific S3 bucket named 'company-reports'. The policy should allow s3:GetObject and s3:ListBucket actions only for that bucket."}, {'task': 'Write a regular expression that validates AWS S3 bucket names according to AWS naming rules: 3-63 characters long, only lowercase letters, numbers, hyphens, and periods, must start and end with a letter or number, and cannot contain consecutive periods or period-hyphen combinations.'}]


In [6]:
with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)